# 11 · Advanced Backtracking: N-Queens & Rat in a Maze

**Backtracking template:**
```
solve(state):
    if state is complete: save it; return
    for each option:
        if option is valid:
            make the move
            solve(new state)
            undo the move          ← backtrack
```
The code is exactly my original; I added explanations/comments and ⚠️ notes where needed.

**Topics:** N-Queens (brute-force check) · N-Queens (optimal with hashing) · Rat in a Maze

## 1. N-Queens
Place `n` queens on an `n × n` board so no two attack each other (no shared row, column or diagonal). Return all boards.

**Idea:** one queen per **column**, left to right. For column `col` try every `row`; if the square is safe, place a queen and recurse on `col + 1`; afterwards remove it (backtrack).
Since columns are filled left to right, `isSafe` only checks squares to the **left**: the same row, the upper-left diagonal and the lower-left diagonal.

**Time:** about O(n! · n) because each safety check scans the board.

In [ ]:

# N Queens
class Solution:


  def solve(self,col,board,ans,n):
    # all n queens placed → store a copy of the board
    if col==n:
      ans.append(list(board))
      return
    for row in range(n):
      # is this square attacked by an earlier queen?
      if self.isSafe(row,col,board,n):
        # place the queen (strings are immutable, so rebuild the row)
        board[row]=board[row][:col]+"Q"+board[row][col +1:]
        # next column
        self.solve(col+1,board,ans,n)
        # remove the queen (backtrack)
        board[row]=board[row][:col]+"."+board[row][col+1:]
  def isSafe(self,row,col,board,n):
    duprow=row
    dupcol=col
    # check the upper-left diagonal
    while row>=0 and col>=0:
      if board[row][col]=="Q":
        return False
      row-=1
      col-=1
    col=dupcol
    row=duprow
    # check the same row to the left
    while col>=0:
      if board[row][col]=="Q":
        return False
      col-=1

    col=dupcol
    row=duprow
    # check the lower-left diagonal
    while row<n and col>=0:
      if board[row][col]=="Q":
        return False
      row+=1
      col-=1
    return True
  def solveNQueens(self, n: int) -> list[list[str]]:
          ans=[]
          board=["."*n for _ in range(n)]
          self.solve(0,board,ans,n)
          return ans
n=4
obj=Solution()
print(obj.solveNQueens(n))




[['..Q.', 'Q...', '...Q', '.Q..'], ['.Q..', '...Q', 'Q...', '..Q.']]


## 2. N-Queens: optimal with hashing
*(my original notes)*

#optimal using hashing
#horizontal to check


*   **Horizontal** = **len(n)** with horizontal empty spaces in a list from 0 to n indexes


---
# Lower Diagonal

*   **Lower Diagonal** : **len(2n-1)** the len of the list be 2*n -1 . the empty list sholud be 0 t0 2n-1
for check we use ** Row + Col **
* For example: Row= 6 , Col=3
* =>(Row+Col)
* => (6+3)
* => 9
* that  we have to place 9 in [6]
[3] = 9

---
# Upper Diagonal

*  **Upper Diagonal** : **len (2n-1)** Same Length of list but alloting the numbers be like this  **(n-1) + (col-row
)**
* For example : row = 6,
col=3,
n=10.
* => (10-1)+(3-6)
* => (9) + (-3)
* => 6
* that we have to place 6 in that [row][col] i,e., [6][3]=6


### The same idea, explained more clearly
Instead of scanning the board, keep three arrays of flags and check **in O(1)**:

| Array | Size | Tracks | Index for `(row, col)` |
|---|---|---|---|
| `leftrow` | `n` | rows already used | `row` |
| `lowerDiagonal` | `2n−1` | diagonals going ↙/↗ | `row + col` |
| `upperDiagonal` | `2n−1` | diagonals going ↖/↘ | `(n−1) + (col − row)` |

**Why the formulas work:** all squares on one ↗ diagonal share the same `row + col` (0 … 2n−2). All squares on one ↘ diagonal share the same `col − row` (from `−(n−1)` to `n−1`); adding `n−1` shifts it to `0 … 2n−2`.
Example: `row=6, col=3, n=10` → `row+col = 9` and `(n−1)+(col−row) = 9 + (3−6) = 6`.

When a queen is placed set the three flags to `1`; when backtracking set them back to `0`. **Time:** O(n!) · **Space:** O(n) for the flag arrays.

In [ ]:
class Solution:
    def solve(self, col, board, ans, leftrow, upperDiagonal, lowerDiagonal, n):
        # Base case: placed all queens successfully
        # placed all queens successfully
        if col == n:
            ans.append(board[:])  # Make a copy of current board
            return

        # Try each row in current column
        for row in range(n):
            # O(1) safety check using our tracking arrays
            if (
                # row is free
                leftrow[row] == 0
                # ↗ diagonal is free
                and lowerDiagonal[row + col] == 0
                # ↘ diagonal is free
                and upperDiagonal[n - 1 + col - row] == 0
            ):
                # Place queen and update all tracking arrays
                board[row] = board[row][:col] + "Q" + board[row][col + 1 :]
                # place queen: mark row and both diagonals
                leftrow[row] = 1
                lowerDiagonal[row + col] = 1
                upperDiagonal[n - 1 + col - row] = 1

                # Recurse to next column
                self.solve(
                    col + 1, board, ans, leftrow, upperDiagonal, lowerDiagonal, n
                )

                # Backtrack: remove queen and reset tracking arrays
                board[row] = board[row][:col] + "." + board[row][col + 1 :]
                # backtrack: free row and both diagonals again
                leftrow[row] = 0
                lowerDiagonal[row + col] = 0
                upperDiagonal[n - 1 + col - row] = 0

    def solveNQueens(self, n: int) -> list[list[str]]:
        ans = []
        board = ["." * n for _ in range(n)]

        # Initialize tracking arrays
        # one flag per row
        leftrow = [0] * n  # Track occupied rows
        # 2n-1 diagonals in each direction
        upperDiagonal = [0] * (2 * n - 1)  # Track upper diagonals
        lowerDiagonal = [0] * (2 * n - 1)  # Track lower diagonals

        self.solve(0, board, ans, leftrow, upperDiagonal, lowerDiagonal, n)
        return ans
obj=Solution()
obj.solveNQueens(4)

[['..Q.', 'Q...', '...Q', '.Q..'], ['.Q..', '...Q', 'Q...', '..Q.']]

### Rat in a Maze Problem
* For this problem we will use **Lexicographically smallest order**
that is **Down -> Left -> Right -> Up**



## 3. Rat in a Maze
A rat starts at the top-left `(0,0)` and must reach the bottom-right `(n−1,n−1)` of an `n×n` grid. `1` = open cell, `0` = blocked. It may move **Down, Left, Right, Up** and may not visit a cell twice on the same path. Return **all** paths; trying the moves in the order **D → L → R → U** gives the paths already in **lexicographically smallest order**.

**Idea:** mark the current cell visited, try the four directions (inside the grid, open, not visited → recurse with the move letter added), then unmark the cell (**backtrack**) so other paths can use it.

> ⚠️ **Note:** this code prints only `['DDRDRR']`, but `DRDDRR` is also a valid path ((0,0)→(1,0)→(1,1)→(2,1)→(3,1)→(3,2)→(3,3)). The cell is unmarked (`vis[i][j]=0`) only inside a direction branch that actually runs, so a **dead-end cell** (no valid move) stays marked as visited forever and blocks later paths. Unmarking once after all four direction checks would fix it.

In [ ]:
# Rat in Maze

class Solution:
  def findpathhelper(self,i:int,
                     j:int,
                     a : list[list[int]],
                     n : int ,
                     ans : list[str],
                     move :str ,
                     vis : list[list[int]]):
    # reached the destination → store the path
    if i==n-1 and j== n-1:
      ans.append(move)
      return
    # mark the current cell
    vis[i][j]=1
    # down
    # can we move down?
    if i+1<n and not vis[i+1][j] and a[i+1][j]==1:
      vis[i][j]=1
      # go down, append 'D' to the path
      self.findpathhelper(i+1,j,a,n,ans,move+'D',vis)
      # unmark (backtrack) — only runs if this branch was taken
      vis[i][j]=0
    # left
    if j-1 >=0 and not vis[i][j-1] and a[i][j-1]==1:
      vis[i][j]=1
      self.findpathhelper(i,j-1,a,n,ans,move+'L',vis)
      vis[i][j]=0
    # right
    if j+1 <n and not vis[i][j+1] and a[i][j+1]==1:
      vis[i][j]=1
      self.findpathhelper(i,j+1,a,n,ans,move+'R',vis)
      vis[i][j]=0
    # up
    if i-1 >=0 and not vis[i-1][j] and a[i-1][j]==1:
      vis[i][j]=1
      self.findpathhelper(i-1,j,a,n,ans,move+'U',vis)
      vis[i][j]=0

  def ratMaze(self,matrix: list[list[int]])-> list[str]:
    n=len(matrix)
    ans=[]
    vis=[[0 for _ in range(n)]for _ in range(n)]
    # no path if the start is blocked
    if matrix[0][0] == 1:
      self.findpathhelper(0,0,matrix,n,ans,'',vis)
    return ans
obj=Solution()
matrix=[[1,0,0,0],[1,1,0,1],[1,1,0,0],[0,1,1,1]]
print(obj.ratMaze(matrix))

['DDRDRR']


---
### ✅ Key takeaways
- Backtracking = **choose → explore → un-choose**.
- N-Queens: one queen per column; `row`, `row+col`, `(n−1)+col−row` identify the row and both diagonals in O(1).
- Rat in a Maze: mark on the way in, **always unmark on the way out**.